# 02 - Test Data EDA: Simultaneous EEG-fNIRS Dataset

This notebook explores the characteristics of the held-out test dataset, **OpenNeuro ds004514**, which consists of real, unmodified noisy fNIRS recordings acquired simultaneously with EEG. This dataset is crucial for evaluating the denoiser against an independent physiological ground truth: the EEG gamma band power envelope.

**Phase 2 Objective:** Obtain real, unmodified noisy fNIRS recordings from subjects with concurrent EEG, to use as the held-out test set. No synthetic noise is involved at any point in testing.

The primary goals of this EDA are:
1. Load and inspect the preprocessed data generated by `src/data/preprocess_test.py`.
2. Visualize the raw fNIRS (HbO, HbR) signals alongside their spatially-matched EEG gamma envelopes.
3. Analyze the distribution of baseline Pearson correlations between the raw fNIRS and EEG signals. This is the metric our model will be evaluated on improving (Δr).

In [ ]:
# Standard library imports
from pathlib import Path
import logging

# Third-party imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# Configure plotting style
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12

## 1. Configuration and Paths

Define the directory where the preprocessed simultaneous EEG-fNIRS `.npz` files are stored. These files are the output of `src/data/preprocess_test.py`.

In [ ]:
# Path to the directory containing preprocessed test .npz files (output of preprocess_test.py)
PROCESSED_TEST_DIR = Path("../data/processed_test")

print(f"Processed test data directory: {PROCESSED_TEST_DIR}")

## 2. Load a Sample Subject's Data

We'll load one of the `.npz` files generated by `preprocess_test.py` to inspect its contents. Each file contains raw HbO/HbR, the corresponding EEG gamma envelope, and initial correlations.

In [ ]:
npz_files = list(PROCESSED_TEST_DIR.glob("*.npz"))

if not npz_files:
    logging.error(f"No .npz files found in {PROCESSED_TEST_DIR}. Please run src/data/preprocess_test.py first.")
    raise FileNotFoundError(f"No .npz files found in {PROCESSED_TEST_DIR}")

sample_npz_path = npz_files[0] # Load the first available subject
print(f"Loading sample data from: {sample_npz_path}")

sample_data = np.load(sample_npz_path, allow_pickle=True)

print("\nKeys in the loaded .npz file:")
for key in sample_data.keys():
    print(f"  - {key}")

# Extract data for easier access
hbo = sample_data['hbo']
hbr = sample_data['hbr']
eeg_envelope = sample_data['eeg_envelope']
sfreq = sample_data['sfreq']
ch_names = sample_data['ch_names']
r_raw_hbo = sample_data['r_raw_hbo']
r_raw_hbr = sample_data['r_raw_hbr']
subject_id = sample_data['subject_id']

print(f"\nSubject ID: {subject_id}")
print(f"Sampling Frequency: {sfreq:.2f} Hz")
print(f"Number of fNIRS channels: {len(ch_names)}")
print(f"HbO data shape: {hbo.shape}")
print(f"HbR data shape: {hbr.shape}")
print(f"EEG envelope data shape: {eeg_envelope.shape}")

## 3. Visualize Raw fNIRS and EEG Gamma Envelope

Let's plot the raw HbO, HbR, and the corresponding EEG gamma band power envelope for a selected channel. This visualization helps understand the baseline relationship between hemodynamic and electrophysiological signals.

In [ ]:
ch_idx = 0 # Select the first channel for visualization
selected_ch_name = ch_names[ch_idx]

time_points = np.arange(hbo.shape[1]) / sfreq

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

axes[0].plot(time_points, hbo[ch_idx], label='Raw HbO', color='red')
axes[0].set_title(f'Raw HbO (Channel: {selected_ch_name})')
axes[0].set_ylabel('Concentration Change')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.6)

axes[1].plot(time_points, hbr[ch_idx], label='Raw HbR', color='blue')
axes[1].set_title(f'Raw HbR (Channel: {selected_ch_name})')
axes[1].set_ylabel('Concentration Change')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.6)

axes[2].plot(time_points, eeg_envelope[ch_idx], label='EEG Gamma Envelope', color='green')
axes[2].set_title(f'EEG Gamma Band Power Envelope (Matched to {selected_ch_name})')
axes[2].set_ylabel('Normalized Power')
axes[2].set_xlabel('Time (s)')
axes[2].legend()
axes[2].grid(True, linestyle='--', alpha=0.6)

plt.suptitle(f'Raw fNIRS and Matched EEG Gamma Envelope for Subject {subject_id}', fontsize=18, y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.show()

## 4. Baseline Correlations (Raw fNIRS vs. EEG Gamma)

The `preprocess_test.py` script already computes the Pearson correlation between the raw HbO/HbR and the matched EEG gamma envelope for each channel. These are our baseline correlations before any denoising.

In [ ]:
print(f"Baseline Pearson Correlation for {selected_ch_name} (Subject: {subject_id}):")
print(f"  Raw HbO vs. EEG Gamma: {r_raw_hbo[ch_idx]:.3f}")
print(f"  Raw HbR vs. EEG Gamma: {r_raw_hbr[ch_idx]:.3f}")

plt.figure(figsize=(12, 6))
sns.histplot(r_raw_hbo, bins=20, kde=True, color='red', label='Raw HbO-EEG Correlation', alpha=0.7)
sns.histplot(r_raw_hbr, bins=20, kde=True, color='blue', label='Raw HbR-EEG Correlation', alpha=0.7)
plt.title(f'Distribution of Raw fNIRS-EEG Gamma Correlations (Subject: {subject_id})')
plt.xlabel('Pearson Correlation Coefficient (r)')
plt.ylabel('Number of Channels')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print(f"\nMean raw HbO-EEG correlation: {np.mean(r_raw_hbo):.3f} ± {np.std(r_raw_hbo):.3f}")
print(f"Mean raw HbR-EEG correlation: {np.mean(r_raw_hbr):.3f} ± {np.std(r_raw_hbr):.3f}")

## 5. Overall Correlation Distribution Across All Subjects

To get a broader picture, let's aggregate the raw correlations from all processed test subjects.

In [ ]:
all_r_hbo = []
all_r_hbr = []

print("Aggregating correlations from all subjects...")
for npz_path in npz_files:
    data = np.load(npz_path, allow_pickle=True)
    all_r_hbo.extend(data['r_raw_hbo'])
    all_r_hbr.extend(data['r_raw_hbr'])

plt.figure(figsize=(12, 6))
sns.histplot(all_r_hbo, bins=30, kde=True, color='red', label='Raw HbO-EEG Correlation', alpha=0.7)
sns.histplot(all_r_hbr, bins=30, kde=True, color='blue', label='Raw HbR-EEG Correlation', alpha=0.7)
plt.title('Overall Distribution of Raw fNIRS-EEG Gamma Correlations (All Subjects)')
plt.xlabel('Pearson Correlation Coefficient (r)')
plt.ylabel('Number of Channel-Time Series Pairs')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print(f"\nOverall Mean raw HbO-EEG correlation: {np.mean(all_r_hbo):.3f} ± {np.std(all_r_hbo):.3f}")
print(f"Overall Mean raw HbR-EEG correlation: {np.mean(all_r_hbr):.3f} ± {np.std(all_r_hbr):.3f}")

## Conclusion

This EDA provides a foundational understanding of the test dataset. We've visualized the raw fNIRS signals alongside the EEG gamma envelope and examined their baseline correlations. These baseline correlations will serve as the reference against which the denoised fNIRS signals will be compared in Phase 5, allowing us to quantify the improvement (Δr) achieved by the denoising model.